# Module 4 — The FUR Regulon: Literature to Motif

**Lab context:** *E. coli* K-12 | bowtie2 | MEME Suite | Biopython | FUR transcription factor

---

This module connects a published dataset to a motif analysis pipeline:

1. **Understanding the literature** — read a real ChIP-exo paper and extract biological context.
2. **RNA-seq paired-end alignment** — the foundational read-alignment step (extends Module 3), applied to the study's transcriptome data.
3. **Motif discovery with MEME** — given a set of genomic sequences, MEME finds overrepresented patterns.
4. **Sequence extraction with Biopython** — pull binding-site sequences from the *E. coli* K-12 genome by coordinate.
5. **TSS distance analysis** — compute distances between binding sites and transcription start sites.

---

**Learning objectives:**
- Understand a real ChIP-exo study and the data it produced
- Use Biopython to extract genomic sequences by coordinate from a public dataset
- Run MEME and interpret motif output (E-value, logo shape)
- Download and align **paired-end RNA-seq** reads with bowtie2, and turn them into a GFF track
- Compute TSS distances as a downstream regulatory analysis


## Background: Transcription Factors, FUR, and Motifs

### What is a transcription factor?

A **transcription factor (TF)** is a protein that binds to specific short DNA sequences near a gene and controls whether that gene gets expressed. Think of it as a light switch wired into the chromosome: when the TF binds, it turns the gene on (or off). Bacteria like *E. coli* use hundreds of TFs to respond to changing conditions — temperature, nutrients, stress.

### What is FUR?

**FUR** (Ferric Uptake Regulator) is one of the most-studied TFs in *E. coli*. Its job is to sense how much iron is available in the cell and adjust gene expression accordingly:

- **Iron-replete (plenty of iron):** FUR binds its target sites and *represses* iron-uptake genes (no need to import more iron)
- **Iron-depleted (low iron):** FUR releases its targets, allowing iron-uptake genes to be expressed

Iron is essential for most cellular processes but toxic in excess — FUR is the cell's main iron thermostat. It controls roughly 90 genes in *E. coli*.

### What is a regulon?

A **regulon** is the complete set of genes controlled by one TF. The FUR regulon = all genes whose expression is directly regulated by FUR binding. Mapping a regulon requires knowing *where* FUR binds — which is exactly what ChIP-exo (Module 3) measures.

### What is a DNA binding motif?

When a TF binds DNA, it doesn't land randomly — it recognizes a specific short sequence pattern, typically 15–20 bp. This pattern is the TF's **binding motif**. For FUR it's called the **Fur box**. If you look at all the genomic sites where FUR binds, you expect to find the same short sequence pattern repeated at each one.

**MEME** is the tool that discovers this pattern automatically: it takes a set of sequences (your ChIP-exo peaks) and finds the motif that appears more often than chance would predict.

---

## 1. The FUR Regulon — Seo et al. 2014

The FUR binding site data used in this module comes from a real published study:

> **Seo SW, Kim D, Latif H, O'Brien EJ, Szubin R, Palsson BO (2014)** — *Deciphering Fur transcriptional regulatory network highlights its complex role beyond iron metabolism in Escherichia coli* — ***Nature Communications*** 5:4910.
> PMID: 25222563 | PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC4167408/ | DOI: 10.1038/ncomms5910 | GEO: **GSE54901**

The study used ChIP-exo to map Fur-binding sites genome-wide across different iron conditions.

**Two different things live in two different places — this matters for the exercises:**
- **Raw ChIP-exo signal** is deposited at GEO (**GSE54901**), as per-base *coverage tracks* (one row per genomic position). This is what you align to and visualize — it is **not** a list of binding sites.
- **The discrete binding-site coordinates** (the 144 called peaks, with their target genes and signal strengths) are in the paper's **Supplementary Data** table (an Excel file), *not* on GEO. This table is the input to your MEME analysis in Exercise 5.


### Exercise 1 — Understand the paper and its data

Read Seo et al. 2014 (PMC4167408) — or use Claude Code to help you understand it — and write a **3-sentence summary** in the cell below covering:
- What experiment was done, in what organism, and under what conditions
- How many FUR binding sites were identified and what regulatory modes (categories) were described
- Where the discrete binding-site coordinates are found (which supplementary table), and — separately — what kind of data GEO GSE54901 actually holds

In Exercise 5 you will download the **supplementary binding-site table** from the paper and use its coordinates as the input to motif discovery.


> **Answer**
>
> 1. ChIP-exo로 FUR의 binding site들을 찾아 FUR regulon들의 조절 메커니즘을 탐색했다; 
>   E. coli K-12 MG1655; iron-replete (FeCl2) 환경과, iron-starved (dipyridyl, iron chelator) 환경에서, 
>   Wild-type과 fur knock-out된 E.coli들의 Fur binding 상태를 비교했다.
> 
> 2. 119개 FUR binding site들을 식별하였고, 3가지 regulatory mode들이 설명되었다: 
>   Holo-Fur repression, Holo-Fur activation, Apo-Fur activation
>
> 3. binding-site coordinates: Supplementary Data 1 (41467_2014_BFncomms5910_MOESM1073_ESM.xlsx),
>   GEO GSE54901 holds: Fe 1, Fe 2, DPD 1, DPD 2가 각각 있는 환경에서 전사인자 Fur, RpoB의 ChIP-exo 데이터 및 WT와 fur KO strain에서, 각각 Fe 1, Fe 2, DPD 1, DPD 2가 있는 환경에서의 RNA-seq 데이터


## 2. RNA-seq: Paired-End Alignment with `bowtie2`

So far you've worked with **ChIP-exo** data (single-end). The same Fur study also produced **RNA-seq** data — a *second data type* and your first **paired-end** alignment.

Paired-end sequencing reads **both ends** of each DNA fragment, producing two FASTQ files per sample (`_1` = R1, `_2` = R2). `bowtie2` aligns them together (`-1`/`-2`), using the known fragment-size range to place reads more accurately than single-end. In this section you'll download an RNA-seq sample, align it paired-end, and turn it into a GFF — the same pipeline shape as Module 3, but for a new data type.

**Data:** the RNA-seq sub-series of the Fur study is **GEO GSE54900** (same paper, *Nature Communications* 5:4910). In the original lab workflow the paired-end sample `SRR1168133` was aligned with `bowtie2 --very-fast -X 1000 -3 3` (paired-end), then sorted and converted to GFF with `makegff.py`.


### Exercise 2 — Run a paired-end RNA-seq alignment

**Step 1 — Get a paired-end RNA-seq sample.**
Use Claude Code to navigate to the RNA-seq series and pick one paired-end sample:
```
https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE54900
```
Ask Claude Code to help you:
1. Find a **paired-end** RNA-seq run in GSE54900 and get its SRR accession.
2. Download it with `fastq-dump --split-files <SRR>` (or `fasterq-dump`) — paired-end gives you **two** files, `<SRR>_1.fastq` and `<SRR>_2.fastq`.

**Step 2 — Align paired-end.**
Reuse the *E. coli* K-12 bowtie2 index you built in Module 3. Run a **paired-end** alignment — note the `-1`/`-2` inputs (not a single `-U`):
```bash
bowtie2 --very-fast -X 1000 -3 3 -p 2 \
    -x <index_prefix> \
    -1 <SRR>_1.fastq -2 <SRR>_2.fastq \
    -S <SRR>.sam
```
Then `samtools view -bS` → `samtools sort` → `samtools index`, exactly as in Module 3.

**Step 3 — Make a GFF for the RNA-seq data.**
Run your `makegff.py` from Module 3 on the sorted BAM to produce a coverage GFF (this is RNA-seq signal, so both strands are meaningful):
```bash
python makegff.py --separate_strand <SRR>.sorted.bam rnaseq.gff
```
This GFF can be loaded into **MetaScope** alongside the gene annotation to visualize transcription — the same way you visualize the ChIP-exo track.

**Step 4 — Explain the flags.** In the Answer cell, write a **one-sentence explanation for each** flag below (focus on *why*, not just mechanics), drawing on what you saw when you ran the alignment:
- `-X 1000` (maximum fragment/insert size — a paired-end-only flag)
- `-3 3` (3′ trimming)
- `--no-mixed` and `--no-discordant` (paired-end concordance)

Also answer: **why is `-X` meaningless for the single-end ChIP-exo alignment in Module 3, but important here?** And: what would go wrong downstream if you forgot `--no-discordant`?

> **Single-end vs paired-end:** Module 3 aligned ChIP-exo reads with a single `-U` input. Here you use `-1`/`-2`. If you accidentally pass paired files to `-U`, bowtie2 treats them as unpaired and every `-X`/concordance flag is silently ignored — a common and quiet mistake.

Write your working pipeline in the code cell below (or run it in the terminal and paste the commands here).


In [ ]:
%%bash
export PATH="/opt/sratoolkit/bin:$PATH"
# Paired-end RNA-seq pipeline (GSE54900) — fill in the commands after generating them with
# Claude Code. This is a bash cell: the FIRST line MUST be %%bash, or Jupyter will try to
# run these shell commands as Python and fail with a SyntaxError.
# Run from the repo root (the notebook's working directory); reads/outputs live in data/reference/.

# Step 1: Download the paired-end RNA-seq sample (fastq-dump --split-files -> _1.fastq / _2.fastq)
fasterq-dump --split-files SRR1168133 -O ../data/reference/

# Step 2: Align paired-end reads, reusing the Module 3 bowtie2 index (-1/-2, not -U)
bowtie2 --very-fast -X 1000 -3 3 -p 2 --no-mixed --no-discordant \
    -x ../data/reference/SRR1168122_index \
    -1 ../data/reference/SRR1168133_1.fastq -2 ../data/reference/SRR1168133_2.fastq \
    -S ../data/reference/SRR1168133.sam

# Step 3: SAM -> sorted, indexed BAM (samtools view / sort / index)
samtools view -bS ../data/reference/SRR1168133.sam -o ../data/reference/SRR1168133.bam
samtools sort -@ 1 ../data/reference/SRR1168133.bam -o ../data/reference/SRR1168133_sorted.bam
samtools index ../data/reference/SRR1168133_sorted.bam

# Step 4: Run makegff.py on the sorted BAM to make the RNA-seq coverage GFF
python makegff.py --separate_strand --log-transform ../data/reference/SRR1168133_sorted.bam ../data/reference/rnaseq.gff

> **Answer**
>
> **`-X 1000`:** concordant로 카운트될 리드 쌍의 최대 길이를 1000bp로 정한다. RNA-seq fragment들의 길이 분포가 1kb 미만인 것으로 알려져 있으므로, 이 최댓값을 설정하여 bowtie2가 비현실적으로 서로 멀리 떨어진 쌍들은 제외하도록 한다.
>
> **`-3 3`:** alignment 전 모든 리드의 3' 말단의 3bp를 잘라낸다. Illumina의 시퀀싱 품질이 3' 말단으로 갈수록 떨어지기 때문에, 낮은 품질의 서열은 mismatch/misalignment를 일으킬 수 있다.
>
> **`--no-mixed` / `--no-discordant`:** 출력이 properly-paired (concordant) alignment만 포함하도록 하여, 한쪽 쌍만 매핑되거나, 틀린 방향/간격 (disconcordant)으로 매핑된 리드들을 제거한다. downstream에는 high-confidence인 쌍들만 들어가게 된다.
>
> **Why `-X` matters here but not in Module 3 (single-end):** -X는 한 쌍의 두 짝 사이의 거리의 범위를 설정한다. ChIP-exo 리드(single-end)를 처리할 때는 의미 없지만, paired-end alignment는 concordancy를 정의하기 위해 짝 사이의 거리에 의존하기 때문에 필수적이다.
>
> **What goes wrong without `--no-discordant`:** Discordant 쌍이 유지되어 정상적인 coverage로 카운트 되면, 짝이 mismapped된 loci/strands에서 depth가 과대 count될 수 있다. 이로 인해 생기는 low-level coverage는 MetaScope에서 가짜 signal을 일으킬 수 있다.


## 3. How MEME Works

### What problem does MEME solve?

You have 90 genomic sequences — one around each FUR binding site. If FUR binds a specific sequence pattern (the Fur box), that pattern should appear in most of your sequences. But you don't know in advance *what* the pattern looks like or exactly *where* in each sequence it sits.

**MEME** (Multiple Em for Motif Elicitation) finds overrepresented sequence patterns in a set of input sequences without knowing what to look for. It uses an algorithm called Expectation-Maximization (EM) to iteratively refine a motif model until it fits the data as well as possible.

### What is an E-value?

The **E-value** answers: *"How many times would I expect to find a motif this good by random chance in a dataset this size?"*

- E-value of 10⁻²⁰ → extremely unlikely by chance → strong, real motif
- E-value of 1.5 → you'd expect to see this pattern 1–2 times by chance → likely noise

For a real TF binding motif like the Fur box, expect E-values well below 10⁻¹⁰. If your top motif has an E-value above 0.01, something may be wrong with your input sequences (wrong coordinates, wrong organism, mixed conditions).

Use Claude Code to understand how the EM algorithm works and what the E-value measures before running MEME.

### Exercise 3 — Understand MEME in your own words

Use Claude Code to explore the MEME algorithm and E-value interpretation. Then write a **3-sentence summary in your own words** — not copied from Claude's output, but re-stated in your own understanding.

> Focus on: what problem EM is solving, what E-value measures, and why motif width matters.

> **Answer**
>
> MEME 알고리즘은 정답을 모르는 상태에서 입력된 시퀀스 집합 안에서 반복 패턴(motif)를 찾아내는 도구이다.
>
> 그 핵심인 EM(Expectation-Maximization) 알고리즘은, motif의 정확한 위치와 서열을 모르는 상태에서 가장 그럴듯한 motif 모델과 각 서열에서 motif가 시작하는 위치를 동시에 추정하기 위해, E-step(현재 motif 모델을 가정하고, 시퀀스들의 각각의 위치가 motif일 확률을 계산한다)과 M-step(그 확률들로 가중평균을 내서 motif의 position wieght matrix를 다시 추정)을 모델이 더 이상 개선되지 않을 때까지 반복해서 시행한다.
>
> E-value는 발견한 motif가 우연히 나올 것으로 기대되는 개수로, 낮을수록 우연이 아닌 진짜 신호일 가능성이 크고, motif의 폭(width)은, MEME 실행 시 폭을 미리 지정 또는 범위로 지정하는데 폭이 너무 좁으면 실제 결합 motif의 일부만 잡아 우연한 motif가 흔하게 나타날 수 있고(E-value 상승), 너무 넓으면 진짜 신호 주변에 무관한 서열이 섞여 모델이 희석될 수 있다.

### Exercise 4 — Conceptual prediction (answer before asking Claude)

**Without asking Claude Code first:** predict what the MEME output would look like for each of these two inputs:

(a) 10 **random** 200 bp genomic sequences from *E. coli* K-12  
(b) 10 **real FUR binding site** sequences (±50 bp around known sites)

For each case, predict: approximate E-value range, whether a clear motif logo would appear, and what width you would expect.

Write your prediction below. Then verify your reasoning with Claude Code and note where you were right or wrong.

(a) E-value가 0.01보다 높은 20bp 미만의 짧은 motif logo가 나타날 것 같다. (b) E-value가 0.01보다 매우 낮은 19bp 정도의 motif logo가 나타날 것 같다.

---

(a) E-value가 0.01보다 높은 것은 맞다 - 다만 random noise에서는 사실 0.01 근처가 아니라 훨씬 더 높게(E-value >> 1) 나온다. 즉, 전혀 유의미하지 않은 것에 가깝다. width에 대해서는, MEME는 노이즈 데이터에서도 -minw~ -maxw 범위 안에서 뭔가는 결과로 내놓기 때문에, 출력되는 width는 범위 내에서 사실상 임의적이다. random 입력에서 진짜 구분되는 특징은 width가 아니라 motif logo 모양 자체가 flat/애매하다는 것이다. (특정 위치에서 특정 염기가 뚜렷하게 우세하지 않고, 4개 염기가 비슷한 확률로 섞여 보임)

(b) 는 FUR box가 실제로 약 19bp palindromic 서열이라 예측한 방향과 크기가 맞는데, 소수(10개)의 서열이라도 진짜 신호가 있어 EM이 뚜렷한 패턴을 잡으면 E-value가 0.01보다 훨씬 낮게(보통 1e-5 이하) 나온다.

## 4. Extracting Sequences with Biopython

The Fur-binding site coordinates come from the **Supplementary Data table of Seo et al. 2014** (an Excel file you download from the paper in Exercise 5), which lists each site's `ChIP-exo Start` / `ChIP-exo End`. Use Biopython to extract DNA sequences from the *E. coli* K-12 reference FASTA at those coordinates. Handle strand correctly — reverse-complement sequences on the minus strand.


### Exercise 5 — Fetch the data and write the sequence extraction script

**Step 1 — Get the binding-site table with Claude Code:**
The discrete Fur binding sites are in the **Supplementary Data** of the Nature Communications paper, not on GEO. Give Claude Code the article page and ask it to locate and download the supplementary table:

```
https://www.nature.com/articles/ncomms5910
```

Ask Claude Code to:
1. Find the **Supplementary Data** file that contains the Fur ChIP-exo **binding sites** (target gene, `ChIP-exo Start`, `ChIP-exo End`, binding condition, S/N ratio) and download it into `data/reference/`. *(It is an Excel `.xlsx` file — the same one used in the original lab tutorial.)*
2. Download the *E. coli* K-12 reference FASTA into `data/reference/NC_000913.fasta` (same file as Module 3 — skip if already present).

**Step 2 — Read the table.** It has a title row above the real header, so read it with the header on the **second** row:
```python
import pandas as pd
df = pd.read_excel("data/reference/<supplementary_file>.xlsx", header=1)
df = df.dropna(subset=["Peak"])          # keep rows that are actual binding sites
```
Columns include: `Transcription Unit`, `Peak` (a peak **ID label** like `P1`, `P2` — *not* a score), `ChIP-exo Start`, `ChIP-exo End`, `Binding Condition` (`R` = iron-replete, `S` = iron-depleted, `R/S` = both), `S/N ratio` and `Significance (p-value)` (these are the strength/score fields), and `Distance to TSS`. Filter to the **iron-replete** sites with `Binding Condition` containing `R`.

**Step 3 — Write a Biopython script that:**
1. Reads the binding-site coordinates from the table (iron-replete sites).
2. Loads the reference FASTA.
3. Extracts sequences ±50 bp around each binding site center (center = midpoint of `ChIP-exo Start`/`End`).
4. Handles strand correctly if a strand is available (reverse complement for `-`).
5. Writes a FASTA file (`data/reference/fur_sites_for_meme.fasta`) ready for MEME input.

> **Coordinate convention warning:** the table's coordinates are **1-based**. Python string slicing is **0-based**. Off-by-one errors here silently shift every extracted sequence.

> **Genome-version note:** the paper's coordinates were called on the *E. coli* K-12 MG1655 genome. Extract from a reference FASTA consistent with those coordinates, and sanity-check a couple of sequences (e.g. does a known Fur site contain a recognizable Fur box?) before trusting all of them.

Run the script and confirm the output FASTA has about one sequence per binding site before proceeding. For reference, the table has **144** sites total; **143** are bound under the iron-replete condition (82 labelled `R` = iron-replete only, plus 61 labelled `R/S` = bound in both conditions), and only 1 is `S`-only.

Write the final working script in the code cell below.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [9]:
import math
import os
import sys

import pandas as pd
from Bio import SeqIO

XLSX_PATH = "../data/reference/ncomms5910_supplementary_data1.xlsx"
REFERENCE_DIR = "../data/reference"
OUT_FASTA_PATH = "../data/reference/fur_sites_for_meme.fasta"
FLANK = 50
FUR_BOX_CONSENSUS = "GATAATGATAATCATTATC"  # 19bp Fur box (Escolar et al. 1998, J Bacteriol)
FUR_BOX_HIT_THRESHOLD = 0.7


def resolve_reference_fasta(reference_dir=REFERENCE_DIR):
    """data/reference 디렉토리의 E. coli K-12 MG1655 reference FASTA의 경로를 반환하며,
    실제로 어떤 RefSeq 버전을 썼는지 로그로 남긴다.

    Seo et al. 2014의 ChIP-exo 좌표는 NC_000913.2 기준으로 불렸다. NC_000913.3은
    .2 대비 몇 군데 서열 교정(indel)이 반영되어 있어서, .3을 쓰면 그 교정 지점
    이후의 좌표가 밀릴 수 있다 (아래 positive_control()로 실제 검증: .3에서는
    알려진 강한 Fur 사이트 5개 중 0개 통과, .2에서는 5개 전부 통과).
    .2가 있으면 우선 사용하고, .3으로 fallback할 경우 경고를 띄운다.
    """
    candidates = ["NC_000913.2.fasta", "NC_000913.fasta", "NC_000913.3.fasta"]
    for name in candidates:
        path = os.path.join(reference_dir, name)
        if os.path.isfile(path):
            print(f"[genome] 사용 중인 reference FASTA: {path}")
            if "913.3" in name:
                print(
                    "[genome] 경고: 논문 좌표는 NC_000913.2 기준입니다. NC_000913.3에는 "
                    "이후 교정된 indel이 있어 좌표가 밀릴 수 있습니다. 아래 "
                    "positive_control() 결과를 반드시 확인한 뒤 추출 결과를 신뢰하세요."
                )
            return path
    tried = [os.path.join(reference_dir, n) for n in candidates]
    sys.exit(f"Error: reference FASTA not found. Tried: {tried}")


def load_binding_sites(xlsx_path=XLSX_PATH):
    """Seo et al. 2014 Supplementary Data 1을 로드하여 iron-replete subset을 반환한다.

    엑셀파일에서 실제 헤더 위에 제목 행이 있으므로 header=1을 사용하여 둘째 줄부터 읽는다.
    한 행은 모두 NaN인 placeholder 행이므로 dropna(subset=["Peak"])로 제거한다.
    Binding Condition 열에 'R'이나 'R/S'처럼 substring 'R'이 포함된 행을 iron-replete sites로 간주한다.
    단, 이 조건은 이 파일에서만 적용되며, 다른 파일에서 'NR', 'RS', 'Ref' 등 다양한 'R'이 포함될 수 있으므로 주의해야 한다.
    """
    if not os.path.isfile(xlsx_path):
        sys.exit(f"Error: binding-site table not found: {xlsx_path}")
    df = pd.read_excel(xlsx_path, header=1)
    df = df.dropna(subset=["Peak"])
    print(f"Binding sites in table (after dropping placeholder row): {len(df)}")
    df = df[df["Binding Conditiona"].astype(str).str.contains("R")]
    print(f"Iron-replete sites (Binding Condition contains 'R'): {len(df)}")
    return df.reset_index(drop=True)


def find_strand_column(df):
    """표에서 strand 열이 있다면 해당 열의 이름을 반환한다. 해당 열이 없으면 None 반환.

    이름이 정확하게 'Strand' 또는 'strand'인 열만 해당한다.
    """
    for col in ("Strand", "strand"):
        if col in df.columns:
            return col
    return None


def get_strand(row, strand_col):
    """이 행의 strand가 +인지 -인지 판단한다.

    strand 열이 없거나, 열은 있지만 값이 없을때 '+'를 반환한다.
    """
    if strand_col is None or pd.isna(row[strand_col]):
        return "+"
    return "-" if str(row[strand_col]).strip() == "-" else "+"


def site_window(start_1based, end_1based, flank=FLANK):
    """1-based inclusive [start, end]구간의 중심점을 기준으로,
    ±flank 범위를 갖는 0-based Python 슬라이싱 경계를 계산한다.
    
    (slice_start, slice_end, window_start_1based, window_end_1based)를 반환한다.
    """
    start_1based, end_1based = int(start_1based), int(end_1based)
    center = math.floor((start_1based + end_1based) / 2 + 0.5)  # 명시적으로 0.5를 반올림
    window_start_1based = center - flank
    window_end_1based = center + flank
    # 1-based inclusive 위치 p -> 0-based 인덱스는 (p-1)이다.
    # slice_start는 -1이 필요하지만, slice_end는 필요 없다.
    # 왜냐하면 Python 슬라이싱의 끝은 원래 exclusive라서,
    # (0-based 변환의 -1)과 (exclusive 끝의 +1)이 서로 상쇄되기 때문이다.
    slice_start = window_start_1based - 1
    slice_end = window_end_1based
    return slice_start, slice_end, window_start_1based, window_end_1based


def extract_site_sequence(genome_seq, start_1based, end_1based, strand, flank=FLANK):
    """genome_seq에서 binding site 주변 ±flank 구간을 잘라낸다.
    
    (sequence_str, window_start_1based, window_end_1based)를 반환하며,
    strand == '-'이면 reverse complement로 변환한다.
    구간이 chromosome 양 끝을 벗어나면 음수 인덱스로 wrap되는 것을 
    막기 위해, 이 경우엔 None을 반환한다.
    """
    slice_start, slice_end, window_start, window_end = site_window(start_1based, end_1based, flank)
    if slice_start < 0 or slice_end > len(genome_seq):
        # 참고: 사용 중인 genome은 단일 환형 chromosome이지만, flank=50 기준으로
        # 실제 143개 iron-replete peak 전부 경계 문제 없이 추출됨 (skip=0).
        # 따라서 wrap-around 처리는 지금 불필요하다고 판단하여 구현하지 않음.
        return None
    window_seq = genome_seq[slice_start:slice_end]
    if strand == "-":
        window_seq = window_seq.reverse_complement()
    return str(window_seq), window_start, window_end


def build_fasta_header(peak_id, chrom_id, window_start, window_end, strand):
    """추출된 하나의 binding site에 대해 MEME에 입력 가능한, 추적가능한 
    FASTA 헤더를 생성한다.
    
    raw ChIP-exo Start/End가 아닌 실제로 추출된 window 좌표를 사용하여
    별도의 source table 없이도 헤더가 설명할 수 있게 한다.
    FASTA/MEME 파서들이 whitespace를 기준으로 헤더를 나누므로 스페이스를 쓰지 않는다.
    """
    return f"{peak_id}_{chrom_id}:{window_start}-{window_end}({strand})"


def write_fasta(records, out_path):
    """(header, sequence) 튜플 목록을 out_path에 unwrapped FASTA 형식으로 저장한다."""
    with open(out_path, "w") as f:
        for header, seq in records:
            f.write(f">{header}\n{seq}\n")
    print(f"FASTA records written: {len(records)}")
    print(f"Output written to: {out_path}")


def _reverse_complement_str(seq):
    return seq.translate(str.maketrans("ACGTacgt", "TGCAtgca"))[::-1]


def best_fur_box_match(seq, consensus=FUR_BOX_CONSENSUS):
    """양쪽 strand에서 FUR_BOX_CONSENSUS를 슬라이딩하며 가장 잘 맞는 위치를 찾는다.

    (identity_fraction, strand, window_start_offset, matched_subseq)를 반환한다.

    IUPAC degeneracy나 PWM 없이 단순 위치별 일치율만 보는 거친(crude) 체크다.
    MEME을 돌리기 전에 좌표/genome 버전이 완전히 틀린 것 같은 큰 문제를 잡기
    위한 것이지, MEME의 통계적 motif 모델을 대신하는 것은 아니다.
    """
    w = len(consensus)
    best = (0.0, "+", -1, "")
    for strand, candidate in (("+", seq), ("-", _reverse_complement_str(seq))):
        for i in range(len(candidate) - w + 1):
            window = candidate[i : i + w].upper()
            identity = sum(a == b for a, b in zip(window, consensus)) / w
            if identity > best[0]:
                best = (identity, strand, i, window)
    return best


def report_fur_box_hit_rate(df, genome_seq, strand_col, threshold=FUR_BOX_HIT_THRESHOLD):
    """top-S/N 몇 개만이 아니라 추출된 모든 site를 FUR_BOX_CONSENSUS와 비교해서,
    threshold 이상 일치하는 site 수와 전체 identity 분포를 보고한다.

    이렇게 하면 실제 Fur box가 부족한 상황을 MEME 결과가 이상하게 나온 뒤에야
    알아채는 게 아니라, 미리 눈에 띄게 만들 수 있다.
    """
    scores = []
    for _, row in df.iterrows():
        strand = get_strand(row, strand_col)
        result = extract_site_sequence(genome_seq, row["ChIP-exo Start"], row["ChIP-exo End"], strand)
        if result is None:
            continue
        seq, _, _ = result
        identity, _, _, _ = best_fur_box_match(seq)
        scores.append(identity)

    hits = sum(s >= threshold for s in scores)
    scores.sort()
    print(
        f"\nFur-box consensus check ({FUR_BOX_CONSENSUS}, {len(FUR_BOX_CONSENSUS)}bp): "
        f"{hits}/{len(scores)} sites reach >={threshold:.0%} identity (best of both strands)"
    )
    print(
        f"Identity distribution: min={scores[0]:.0%}  median={scores[len(scores) // 2]:.0%}  "
        f"max={scores[-1]:.0%}"
    )


def positive_control(df, genome_seq, chrom_id, strand_col, n=5, max_mismatch=5,
                      box_col="Significance (p-value) of similarity to Fur box"):
    """MEME으로 넘기기 전에, 추출 자체가 맞는지 검증한다.

    논문이 스스로 'Fur box와 가장 유사하다'고 평가한(p-value가 가장 작은) 상위 n개
    site를 골라(그 열이 없으면 교과서적으로 잘 알려진 Fur 타깃 목록으로 대체),
    추출된 window 안에 실제로 FUR_BOX_CONSENSUS와 가까운 서열(<= max_mismatch/19bp,
    양쪽 strand 중 더 잘 맞는 쪽)이 있는지 확인한다.

    알려진 강한 site 중 단 하나도 통과하지 못하면 좌표나 genome 버전이 잘못된
    것이므로, MEME으로 넘어가지 말고 여기서 즉시 멈춘다(sys.exit) -- 이 버그를
    나중에 "MEME 결과가 이상하다"는 형태로 뒤늦게 발견하지 않기 위함이다.
    """
    if box_col in df.columns:
        targets = df.copy()
        targets[box_col] = pd.to_numeric(targets[box_col], errors="coerce")
        targets = targets.dropna(subset=[box_col]).sort_values(box_col).head(n)
    else:
        known = {"fepA-entD", "fhuACDB", "feoABC", "mntH", "fecIR"}
        targets = df[df["Transcription Unit"].isin(known)].head(n)

    print(f"\n--- Positive control: {len(targets)} known strong Fur-box sites ---")
    passed = 0
    for _, row in targets.iterrows():
        strand = get_strand(row, strand_col)
        result = extract_site_sequence(genome_seq, row["ChIP-exo Start"], row["ChIP-exo End"], strand)
        if result is None:
            print(f"  {row['Peak']}: out of chromosome bounds")
            continue
        seq, window_start, window_end = result
        identity, match_strand, _, match_seq = best_fur_box_match(seq)
        mismatches = round((1 - identity) * len(FUR_BOX_CONSENSUS))
        ok = mismatches <= max_mismatch
        passed += ok
        print(
            f"  {row['Peak']:5s} {str(row['Transcription Unit']):18s} "
            f"{chrom_id}:{window_start}-{window_end}({strand})  "
            f"best Fur-box = {mismatches}/{len(FUR_BOX_CONSENSUS)} mismatches  "
            f"strand={match_strand}  {match_seq}  -> {'PASS' if ok else 'FAIL'}"
        )
    print(f"--- {passed}/{len(targets)} passed (<= {max_mismatch} mismatches) ---")

    if passed == 0:
        sys.exit(
            "FATAL: 알려진 강한 Fur-box site 어디에서도 인식 가능한 Fur box를 찾지 못했습니다.\n"
            "       좌표 또는 genome 버전(NC_000913.2 vs .3)이 어긋났을 가능성이 큽니다.\n"
            "       MEME으로 넘어가지 마세요 -- 먼저 추출을 고쳐야 합니다."
        )
    elif passed < len(targets):
        print(
            "NOTE: 일부만 통과했습니다. 전면적 실패는 아니지만, 통과하지 못한 site의 "
            "좌표/window를 점검하는 것이 좋습니다."
        )


def main():
    df = load_binding_sites()
    strand_col = find_strand_column(df)
    if strand_col is None:
        """
        binding-site 표에 strand 열이 없는 경우, 모든 site를 '+'로 간주한다.
        이는 이후 결과에 편향을 주지 않는다: 다음 섹션에서 MEME을 -revcomp
        옵션으로 실행하기 때문에, 입력 FASTA의 strand 기록 방식과 무관하게
        양쪽 strand를 모두 탐색하게 된다.
        """
        print(
            "No strand column found in the binding-site table -- defaulting all "
            "sites to '+'. This does not bias downstream results: MEME is run "
            "with -revcomp in the next section, which searches both strands "
            "regardless of how the input FASTA strand was recorded."
        )

    fasta_path = resolve_reference_fasta()
    # Biopython의 SeqIO 모듈로 reference FASTA 파일을 파싱하여 서열 레코드를 
    # 가져와 Biopython의 SeqRecord 객체로 저장한다.
    # E. coli는 단일 chromosome이므로, 파일에는 서열이 하나만 있다고 가정한다.
    genome_record = next(SeqIO.parse(fasta_path, "fasta"))
    # 실제 염기서열(Seq 객체)을 별도 변수로 꺼내둔다.
    genome_seq = genome_record.seq
    # FASTA 헤더의 서열 ID를 저장한다.
    chrom_id = genome_record.id

    records = []
    skipped = 0
    for _, row in df.iterrows():
        strand = get_strand(row, strand_col)
        result = extract_site_sequence(genome_seq, row["ChIP-exo Start"], row["ChIP-exo End"], strand)
        if result is None:
            skipped += 1
            print(f"Skipped {row['Peak']}: window out of chromosome bounds")
            continue
        seq, window_start, window_end = result
        header = build_fasta_header(row["Peak"], chrom_id, window_start, window_end, strand)
        records.append((header, seq))

    write_fasta(records, OUT_FASTA_PATH)
    print(f"Sites skipped (window out of chromosome bounds): {skipped}")

    report_fur_box_hit_rate(df, genome_seq, strand_col)
    positive_control(df, genome_seq, chrom_id, strand_col)


main()


Binding sites in table (after dropping placeholder row): 144
Iron-replete sites (Binding Condition contains 'R'): 143
No strand column found in the binding-site table -- defaulting all sites to '+'. This does not bias downstream results: MEME is run with -revcomp in the next section, which searches both strands regardless of how the input FASTA strand was recorded.
[genome] 사용 중인 reference FASTA: ../data/reference/NC_000913.2.fasta
FASTA records written: 143
Output written to: ../data/reference/fur_sites_for_meme.fasta
Sites skipped (window out of chromosome bounds): 0

Fur-box consensus check (GATAATGATAATCATTATC, 19bp): 40/143 sites reach >=70% identity (best of both strands)
Identity distribution: min=47%  median=63%  max=89%

--- Positive control: 5 known strong Fur-box sites ---
  P82   nan                NC_000913.2:2510730-2510830(+)  best Fur-box = 4/19 mismatches  strand=-  GAATTTGATAATCATTCTC  -> PASS
  P81   mntH               NC_000913.2:2510723-2510823(+)  best Fur-box = 4

## 5. Running MEME

MEME is a command-line tool. The basic invocation is:

```bash
meme input.fasta -dna -oc output_dir -mod zoops -minw 15 -maxw 25 -nmotifs 3
```

Use **Claude Code plan mode** to generate the full command with the flags appropriate for your FUR binding site dataset. Make sure you understand each flag before running.

> **Think first:** FUR binds double-stranded DNA, so a binding motif can appear on either strand. Which MEME flag accounts for this? Write your answer before generating the command.

In [10]:
%%bash
# Think first: FUR는 double-stranded DNA에 결합하므로, binding motif는 양쪽 strand 모두에 나타날 수 있다.
# 입력되는 모든 143개 시퀀스들은 표에서 strand 열이 없었기 때문에 '+' strand로 기록되어 있다.
# 따라서 -revcomp 플래그를 통해 MEME이 reverse strand도 스캔하게 하지 않으면 
# 실제로 minus strand에 있는 Fur box가 MEME에 의해 무관한 것으로 판단될 수 있다.

# -dna:         입력 시퀀스는 DNA임을 명시.
# -oc:          출력 디렉터리 (created/overwritten); meme.txt와 meme.html가 저장될 곳
# -mod zoops:   zero-or-one-occurrence-per-sequence -- 각 101bp window는 하나의 ChIP-exo peak이므로, 
#               시퀀스당 최소 하나의 Fur box가 있을 것으로 기대됨.
# -revcomp:     각 시퀀스의 reverse complement strand도 스캔할 것.(Think first 참조.)
# -minw / -maxw: 15-25 bp 길이 범위 내의 motif를 탐색; 알려진 E. coli Fur box는 ~19bp.
# -nmotifs 3:   최대 3개의 후보 모티프를 보고, significance(E-value) 순
/opt/meme/bin/meme ../data/reference/fur_sites_for_meme.fasta -dna -oc ../data/reference/meme_fur -mod zoops -revcomp -minw 15 -maxw 25 -nmotifs 3


Writing results to output directory '../data/reference/meme_fur'.
BACKGROUND: using background model of order 0
PRIMARY (classic): n 143 p0 143 p1 0 p2 0
SEQUENCE GROUP USAGE-- Starts/EM: p0; Trim: p0; pvalue: p0; nsites: p0,p1,p2
SEEDS: maxwords 14443 highwater mark: seq 143 pos 86
Initializing the motif probability tables for 2 to 143 sites...
nsites = 143s = 5es = 22 30
Done initializing.

seqs=   143, min_w= 101, max_w=  101, total_size=    14443

motif=1
SEED DEPTHS: 2 4 8 16 32 64 128 143
SEED WIDTHS: 15 21 25
em: w=  25, psites= 143, iter=  40 Warning: Cannot convert EPS file to PNG as no install of Image Magick or Ghostscript is usable.

motif=2
SEED DEPTHS: 2 4 8 16 32 64 128 143
SEED WIDTHS: 15 21 25
em: w=  25, psites= 143, iter=  40 Warning: Cannot convert EPS file to PNG as no install of Image Magick or Ghostscript is usable.

motif=3
SEED DEPTHS: 2 4 8 16 32 64 128 143
SEED WIDTHS: 15 21 25
em: w=  25, psites= 143, iter=  40 Warning: Cannot convert EPS file to PNG as no i

### Exercise 6 — Interpret your MEME results

After running MEME:
1. Use Claude Code to help you read the `meme.txt` or `meme.html` output.
2. Then write your **own assessment** — do not just copy Claude's interpretation.

Address all three of the following:
- **E-value:** Is the top motif statistically significant? What does the E-value tell you?
- **Motif logo:** Does the shape match what you expect for a FUR binding site (the "Fur box")? Which positions are most conserved?
- **Biological meaning:** Given your background from Exercise 1 (the published FUR regulon), is this result consistent with the literature? Why or why not?

Write your assessment in the markdown cell below.

> **Answer**
>
> *Your MEME result assessment here.*


## 6. TSS Distance Analysis

A common downstream analysis computes the distance between each transcription factor binding site and the nearest transcription start site (TSS).

### Exercise 7 — TSS distance script

Two data sources are available for this analysis:
- **Binding sites:** the Fur binding-site table you downloaded in Exercise 5 (`ChIP-exo Start`/`End` per site). The table also contains a `Distance to TSS` column for a subset of sites — you can use this to **cross-check** your own computation.
- **Gene annotation:** `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (pre-provided — lab *E. coli* K-12 gene annotation; all rows are `gene` features, so a TSS must be derived from each gene's start/strand: gene start for `+`, gene end for `-`)

Use Claude Code to generate a Python script that:
1. Reads the binding-site coordinates and the annotation GFF.
2. For each Fur binding site, computes the distance to the nearest TSS.

> **Coordinate-space warning:** the binding-site table has no chromosome column — its coordinates are positions on the *E. coli* K-12 genome. The annotation GFF uses seqname `NC_000913`, while the reference FASTA you downloaded is `NC_000913.3`. When you compare binding-site positions to gene positions, make sure both are in the **same coordinate space** (same genome build) — otherwise distances will be systematically off, silently. This is a good place to run a quick sanity check (does a known site land near its known target gene?).
3. Reports: minimum distance, median distance, and fraction of binding sites within 500 bp of a TSS.
4. (Optional) Plots the distance distribution as a histogram.

Write the final working script in the code cell below.

> **Cross-check:** compare your computed distances against the paper's `Distance to TSS` column for the sites that have one. If they disagree by a constant offset, you likely have an off-by-one or a genome-version mismatch.

> **Biological framing:** Based on Exercise 1 (the Seo et al. 2014 paper), what distance distribution do you expect? Write your prediction as a comment at the top of the script.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# TSS distance analysis script.
# Prediction (write before running): ...



## 7. Integrative Visualization in MetaScope

So far this module produced three separate results: an **RNA-seq coverage** track (Section 2), a set of **Fur binding sites** (the table from Exercise 5), and a **motif** (Section 5). Each answers part of the FUR story, but the story only comes together when you see them **on the same axis**.

That is what MetaScope is for. In Module 3 you visualized a single ChIP-exo track. Here you'll overlay **three tracks** and read the regulation directly off the screen:

| Track | File | What it shows |
|-------|------|---------------|
| Gene annotation | `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` | Where the genes are |
| RNA-seq coverage | `rnaseq.gff` (your Exercise 2 output) | How strongly each region is transcribed |
| Fur binding sites | `fur_sites.gff` (you'll build it below) | Where FUR binds |

The biological payoff: at an iron-uptake gene, you should be able to see a **Fur binding site** sitting just upstream **and** ask whether the gene's **RNA-seq signal** is consistent with FUR's role as a repressor.

> As in Module 3, MetaScope runs on **your own computer**, not in the Codespace. Install it from the lab homepage ([sbml-lab.ai](https://sbml-lab.ai)). Produce the GFFs here, download them, then open them locally.

### Exercise 8 — Turn the Binding Sites into a Track

Your RNA-seq track and the annotation are already GFF files. The binding sites, though, are still rows in an Excel table — MetaScope can't read those. Convert them.

Use Claude Code to write a short script that reads the **iron-replete** binding sites (the `R` / `R/S` rows you filtered in Exercise 5) and writes a GFF file `data/reference/fur_sites.gff`, one row per site:

```
NC_000913	furchip	fur_site	<ChIP-exo Start>	<ChIP-exo End>	<S/N ratio>	.	.	name=<Transcription Unit>
```

Two things that matter (both are recurring themes in this course):
- **Chromosome ID must match the other tracks.** Use `NC_000913` in column 1 so this track lands on the *same* chromosome tab as the annotation. (Recall the Module 3 gotcha: MetaScope groups tracks by column 1.)
- **Put the S/N ratio in the score column** so stronger sites can be told apart when you display the track.

Write the script below, run it, and confirm `fur_sites.gff` has roughly one row per iron-replete site (~143).

In [ ]:
# Build fur_sites.gff from the binding-site table (iron-replete sites).
# Reuse the DataFrame you loaded in Exercise 5 (pd.read_excel(..., header=1)).
# Column 1 MUST be NC_000913 so this track lines up with the annotation in MetaScope.



### Exercise 9 — Overlay Three Tracks and Read the Regulation

On your machine, open MetaScope and load **all three** GFFs (`Ctrl/Cmd+O`): the annotation, `rnaseq.gff`, and `fur_sites.gff`.

1. **Pick a Fur-regulated iron gene.** Use `Ctrl/Cmd+F` to search the annotation, or `Ctrl/Cmd+G` to jump to a coordinate. Good candidates (verified in the lab annotation): **`fepA`** (~609–612 kb), the **`entCEBA`** enterobactin cluster (~624–629 kb), **`fhuA`** (~167 kb), or **`sodB`** (~1,733 kb).
2. **Line up the three tracks.** At your chosen gene, check: is there a **Fur binding site** just upstream? What does the **RNA-seq coverage** look like over the gene body?
3. **Note the RNA-seq condition.** FUR represses iron-uptake genes **when iron is replete**. Whether you expect high or low RNA-seq signal depends on the iron condition of the sample you aligned in Exercise 2 — state which condition it is and reason accordingly. (If you're unsure, that's a `/explain` or a quick check of the GSE54900 sample metadata.)
4. **Adjust the display** so the figure is readable: set distinct track colors (`Ctrl/Cmd+Shift+C`) and heights (`Ctrl/Cmd+Shift+H`), and scale the RNA-seq track so its peaks are visible.
5. **Export** (`Ctrl+Shift+E` → PNG, 300 dpi) as `module4_integrative_metascope.png`.

This exported figure is your Module 4 visualization deliverable.

> **Answer**
>
> - Gene chosen (name / locus_tag): 
> - Is there a Fur binding site upstream of it? Position:
> - RNA-seq condition of your sample (iron-replete / iron-depleted):
> - Is the gene's RNA-seq signal consistent with FUR repression **under that condition**? Explain.
> - Attach `module4_integrative_metascope.png`.

### Exercise 10 — Tie It Back to the Motif and the Literature

You now have three lines of evidence at one locus: a **binding site** (the track), a **motif** (Section 5 — is there a Fur box in that site's sequence?), and an **expression level** (the RNA-seq track). Together these are the core of what Seo et al. 2014 reported.

Write a short interpretation (3–5 sentences):
- Does the binding + expression pattern at your gene match FUR acting as an iron-responsive **repressor**?
- Is this **confirmatory** (a known FUR target), or did you land somewhere **unexpected** (FUR's "role beyond iron metabolism," as the paper's title puts it)?
- Use Claude Code to cross-check the gene's function and known FUR regulation — and note anything that *didn't* line up, and how you resolved it.

This is exactly the kind of integrative reading your mini-project (Module 5) will ask you to do independently.

> **Answer**
>
> *(Your 3–5 sentence integrative interpretation — binding + motif + expression, confirmatory vs unexpected, and what you cross-checked.)*

## End of Module 4

You have completed:
- Reading and understanding a real ChIP-exo paper (Seo et al. 2014)
- Downloading and running a **paired-end RNA-seq** alignment, and producing a GFF track from it
- Using Biopython to pull binding-site sequences from a reference genome
- Running MEME and interpreting motif output
- Computing TSS distances as a downstream regulatory analysis
- **Overlaying annotation + RNA-seq + Fur binding sites in MetaScope** and reading the regulation integratively (`module4_integrative_metascope.png`)

---

Run `/log` before closing this session.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module4): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
